## DDL Gold: pf.gold.fact_metricas_diarias  (TABLA DE HECHOS)
## Granularidad: modelo x dia.
## PK: row_hash = md5(model_id|fecha_id|modelo_id)

In [0]:
%sql

DROP TABLE IF EXISTS pf.gold.fact_metricas_diarias;

CREATE TABLE IF NOT EXISTS pf.gold.fact_metricas_diarias (
    row_hash STRING NOT NULL COMMENT 'PK - md5(model_id|fecha_id|modelo_id)',
    fecha_id BIGINT NOT NULL COMMENT 'FK - dim_fecha',
    modelo_id BIGINT NOT NULL COMMENT 'FK - dim_modelo_scd2 (SK version vigente del dia)',
    model_id STRING NOT NULL COMMENT 'BK de trazabilidad',
    org_id STRING NOT NULL COMMENT 'BK - dim_organizacion',
    org_sk BIGINT NOT NULL COMMENT 'FK - dim_organizacion (SK de la org en el dia)',
    task_id BIGINT COMMENT 'FK - dim_task (SK de la tarea en el dia)',
    libreria_id BIGINT COMMENT 'FK - dim_libreria',
    licencia_id BIGINT COMMENT 'FK - dim_licencia',
    likes BIGINT COMMENT 'Snapshot de likes del dia',
    downloads BIGINT COMMENT 'Snapshot de descargas del dia',
    delta_downloads BIGINT COMMENT 'Descargas hoy - descargas dia anterior (0/null el 1er dia)',
    es_primer_dia BOOLEAN COMMENT 'True si el modelo aparece por primera vez',
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP COMMENT 'UTC',
    PRIMARY KEY (row_hash),
    CONSTRAINT fk_fact_fecha FOREIGN KEY (fecha_id) REFERENCES pf.gold.dim_fecha (fecha_id),
    CONSTRAINT fk_fact_modelo FOREIGN KEY (modelo_id) REFERENCES pf.gold.dim_modelo_scd2 (modelo_id),
    CONSTRAINT fk_fact_org FOREIGN KEY (org_sk) REFERENCES pf.gold.dim_organizacion (org_sk),
    CONSTRAINT fk_fact_task FOREIGN KEY (task_id) REFERENCES pf.gold.dim_task (task_id),
    CONSTRAINT fk_fact_libreria FOREIGN KEY (libreria_id) REFERENCES pf.gold.dim_libreria (libreria_id),
    CONSTRAINT fk_fact_licencia FOREIGN KEY (licencia_id) REFERENCES pf.gold.dim_licencia (licencia_id)
)
USING DELTA
TBLPROPERTIES (
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults = 'supported'
)
COMMENT 'Fact Metrica Diaria (granularidad modelo x dia)';

In [0]:
%sql
-- POPULADO (normalmente lo hace ETL/gold/03_gold_fact.py; aqui el SQL canonico)
-- 1) mapea SKs de dims por BK; 2) calcula delta contra dia anterior;
-- 3) inserta por row_hash (idempotente si se vuelve a correr el dia).
CREATE OR REPLACE TEMP VIEW staging_fact AS
WITH sil AS (
    SELECT *
    FROM pf.silver.modelos
    WHERE ingestion_date = (SELECT MAX(ingestion_date) FROM pf.silver.modelos)
)
SELECT
    sil.model_id AS model_id,
    f.fecha_id AS fecha_id,
    dm.modelo_id AS modelo_id,
    sil.org_id AS org_id,
    do2.org_sk AS org_sk,
    do3.task_id AS task_id,
    dl2.libreria_id AS libreria_id,
    dl3.licencia_id AS licencia_id,
    sil.likes AS likes,
    sil.downloads AS downloads
FROM sil
JOIN pf.gold.dim_fecha f
     ON f.fecha = sil.ingestion_date
JOIN pf.gold.dim_modelo_scd2 dm
     ON dm.model_id = sil.model_id AND dm.is_current = TRUE
JOIN pf.gold.dim_organizacion do2
     ON do2.org_id = sil.org_id
LEFT JOIN pf.gold.dim_task do3
     ON do3.pipeline_tag = sil.pipeline_tag
LEFT JOIN pf.gold.dim_libreria dl2
     ON dl2.library_name = sil.library_name
LEFT JOIN pf.gold.dim_licencia dl3
     ON dl3.license_tag = COALESCE(sil.license_tag, 'sin_licencia');


In [0]:
%sql

-- 2) merge idempotente (si re-corres el dia no duplica)
MERGE INTO pf.gold.fact_metricas_diarias AS t
USING (
    SELECT
        MD5(CONCAT_WS('|', s.model_id, s.fecha_id, s.modelo_id)) AS row_hash,
        s.*
    FROM staging_fact s
) AS s
ON t.row_hash = s.row_hash
WHEN MATCHED THEN
    UPDATE SET t.likes = s.likes,
               t.downloads = s.downloads
--    delta_downloads y es_primer_dia se calculan por update/insert separado
WHEN NOT MATCHED THEN
    INSERT (row_hash, fecha_id, modelo_id, model_id, org_id, org_sk,
            task_id, libreria_id, licencia_id, likes, downloads,
            delta_downloads, es_primer_dia, _createdAt)
    VALUES (s.row_hash, s.fecha_id, s.modelo_id, s.model_id, s.org_id, s.org_sk,
            s.task_id, s.libreria_id, s.licencia_id, s.likes, s.downloads,
            NULL, TRUE, CURRENT_TIMESTAMP());

In [0]:
%sql

-- 3) delta_downloads contra el dia anterior (canonico; el runtime vive en
--    ETL/gold/03_gold_fact.py que lo parametriza con el fecha_id de la corrida)
UPDATE pf.gold.fact_metricas_diarias AS t
SET t.delta_downloads =
      t.downloads - COALESCE((
        SELECT prev.downloads
        FROM pf.gold.fact_metricas_diarias prev
        WHERE prev.model_id = t.model_id
          AND prev.fecha_id = CAST(
                DATE_FORMAT(DATE_SUB(TO_DATE(CAST(t.fecha_id AS STRING), 'yyyyMMdd'), 1), 'yyyyMMdd')
                AS BIGINT)
        LIMIT 1), t.downloads)
WHERE t.fecha_id = CAST(DATE_FORMAT(CURRENT_DATE(), 'yyyyMMdd') AS BIGINT);

In [0]:
%sql

-- Verificacion

SELECT * FROM pf.gold.fact_metricas_diarias ORDER BY downloads DESC LIMIT 10;

In [0]:
%sql

SELECT COUNT(*) AS n_rows FROM pf.gold.fact_metricas_diarias;